# Multi-task ResNet18 Attribute Training and Test

Notebook train chung shape/color cho head-tune va last-block fine-tune, dong thoi ho tro test checkpoint co san. Chon `EXECUTION_MODE` o Cell cau hinh. Bat Internet trong Kaggle de clone repo, va attach thu muc `data/` co `image_all/nih_attribute/`, `splits/`, `processed/`; che do test-only can attach them dataset model artifact.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git'
BRANCH = 'FE_Final'
REPO_DIR = Path('/kaggle/working/Multiple-Pill-Recognition-And-Interaction-Safety')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

print('Repository:', REPO_DIR)
print('Branch:', BRANCH)

In [ ]:
# Dung cac goi NumPy/SciPy/Pillow co san cua image Kaggle.
%pip install -q --no-deps ultralytics==8.3.253 pyyaml==6.0.2
# Khong force-reinstall cac goi nen: doi NumPy khi SciPy da duoc kernel nap se gay loi ABI o Cell 5.
numeric_check = subprocess.run(
    [sys.executable, '-c', 'import numpy; import scipy; import sklearn'],
    text=True,
    capture_output=True,
)
if numeric_check.returncode != 0:
    # Chi dung de sua session da bi loi boi cac lan cai dat cu; session moi se khong vao nhanh nay.
    print('Detected broken NumPy/SciPy environment. Repairing compatible numeric packages...', flush=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'uninstall', '-y',
        'numpy', 'scipy', 'scikit-learn'
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2'
    ], check=True)
    print('Numeric packages repaired. Restarting the notebook kernel now...', flush=True)
    import os
    os._exit(0)

# Process con co the doc package moi trong khi kernel hien tai van giu NumPy cu trong RAM.
# Kiem tra lai ngay trong kernel de khong day loi ABI xuong Cell import workflow.
try:
    import numpy as np
    import scipy
    import sklearn
    from scipy.sparse import csr_matrix
except Exception as numeric_kernel_error:
    print(
        'Numeric packages on disk are healthy but this kernel is stale: '
        f'{type(numeric_kernel_error).__name__}: {numeric_kernel_error}',
        flush=True,
    )
    print('Restarting the notebook kernel before workflow imports...', flush=True)
    import os
    os._exit(0)

import PIL
from PIL import Image, ImageDraw
print('Pillow:', PIL.__version__)

# Probe trong process rieng de chua import torch vao kernel notebook.
probe_code = (
    "import torch; "
    "cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (-1,-1); "
    "print('TORCH_PROBE|' + torch.__version__ + '|' + str(torch.version.cuda) + '|' + "
    "str(cap[0]) + '.' + str(cap[1]) + '|' + ','.join(torch.cuda.get_arch_list()))"
)
probe = subprocess.run([sys.executable, '-c', probe_code], text=True, capture_output=True)
probe_line = next(
    (line for line in probe.stdout.splitlines() if line.startswith('TORCH_PROBE|')),
    None,
)

needs_compatible_wheel = probe.returncode != 0 or probe_line is None
if probe_line is not None:
    _, old_torch, old_cuda, capability, arch_text = probe_line.split('|', 4)
    required_arch = 'sm_' + capability.replace('.', '')
    needs_compatible_wheel = required_arch not in arch_text.split(',')
    print('Existing torch:', old_torch, '| CUDA:', old_cuda, '| capability:', capability)
    print('Existing CUDA architectures:', arch_text)

if needs_compatible_wheel:
    print('Current PyTorch wheel does not support this GPU. Installing compatible cu124 wheel...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124'
    ], check=True)
    if 'torch' in sys.modules:
        raise RuntimeError(
            'Da cai PyTorch cu124. Hay Restart Session, sau do Run All de nap wheel moi.'
        )

import torch
import torchvision
if not torch.cuda.is_available():
    raise RuntimeError('Hay bat GPU accelerator trong Kaggle Settings truoc khi train.')

required_arch = 'sm_' + ''.join(map(str, torch.cuda.get_device_capability(0)))
if required_arch not in torch.cuda.get_arch_list():
    raise RuntimeError(
        f'PyTorch {torch.__version__} khong co kernel {required_arch}: ' 
        f'{torch.cuda.get_arch_list()}'
    )

# Smoke test convolution de bat loi kernel ngay tai setup, truoc khi train.
test_conv = torch.nn.Conv2d(3, 4, kernel_size=3).cuda()
test_input = torch.randn(2, 3, 32, 32, device='cuda')
with torch.no_grad():
    test_conv(test_input)
torch.cuda.synchronize()
del test_conv, test_input
torch.cuda.empty_cache()

print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA architectures:', torch.cuda.get_arch_list())
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA convolution smoke test: PASSED')

In [ ]:
# Tu dong tim DATA_ROOT theo cay data/image_all, data/splits va data/processed.
# Khong hard-code slug vi Kaggle co the thay doi so cap thu muc khi mount dataset.
def find_data_root(input_root: Path = Path('/kaggle/input')) -> Path:
    candidates = []
    for image_all_dir in input_root.rglob('image_all'):
        candidate = image_all_dir.parent
        has_images = (image_all_dir / 'nih_attribute/shape').is_dir() and (image_all_dir / 'nih_attribute/color').is_dir()
        has_splits = (candidate / 'splits/nih_attribute/shape').is_dir() and (candidate / 'splits/nih_attribute/color').is_dir()
        if has_images and has_splits and (candidate / 'processed').is_dir():
            candidates.append(candidate)

    unique_candidates = sorted(set(candidates))
    if len(unique_candidates) == 1:
        return unique_candidates[0]
    if not unique_candidates:
        raise FileNotFoundError(
            'Khong tim thay DATA_ROOT. Dataset can co data/image_all/nih_attribute, '
            'data/splits/nih_attribute va data/processed trong /kaggle/input.'
        )
    raise RuntimeError(
        'Tim thay nhieu DATA_ROOT, hay chi dinh mot path: ' +
        ', '.join(str(path) for path in unique_candidates)
    )

def find_model_artifact_dir(input_root: Path = Path('/kaggle/input')) -> Path:
    # Tim dataset model attach vao Kaggle; folder can chua dung bon artifact cua last-block model.
    required = {'best.pt', 'label_mapping.json', 'optimal_thresholds.json', 'model_config.yaml'}
    candidates = sorted({path.parent for path in input_root.rglob('best.pt') if required.issubset({item.name for item in path.parent.iterdir()})})
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        # Fallback de co the chay tren may local voi folder da dong goi trong repo.
        local_artifact_dir = REPO_DIR / 'kaggle_uploads' / 'attribute_resnet18_last_blocks_finetune'
        if required.issubset({item.name for item in local_artifact_dir.iterdir()} if local_artifact_dir.is_dir() else set()):
            return local_artifact_dir
        raise FileNotFoundError('Khong tim thay folder model co best.pt, label_mapping.json, optimal_thresholds.json va model_config.yaml.')
    raise RuntimeError('Tim thay nhieu model artifact folder. Hay chi dinh MODEL_ARTIFACT_DIR: ' + ', '.join(str(path) for path in candidates))

def find_segmentation_weights(input_root: Path = Path('/kaggle/input')) -> Path:
    expected_name = 'yolov11m_seg_mediseg_full_finetune_v1.pt'
    candidates = sorted(input_root.rglob(expected_name))
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f'Khong tim thay {expected_name}. Attach Kaggle Dataset module1_segmentation_artifact.'
        )
    raise RuntimeError('Tim thay nhieu segmentation checkpoint: ' + ', '.join(str(path) for path in candidates))

# Trial nay train lai head va last-block tren crop do Module 1 sinh ra.
EXECUTION_MODE = 'train_head_and_last_blocks'
# EXECUTION_MODE = 'test_existing_last_blocks'
# EXECUTION_MODE = 'train_head_only'

if EXECUTION_MODE not in {'test_existing_last_blocks', 'train_head_and_last_blocks', 'train_head_only'}:
    raise ValueError(f'Unsupported EXECUTION_MODE: {EXECUTION_MODE}')

DATA_ROOT = find_data_root()
SEGMENTATION_WEIGHTS = find_segmentation_weights()
OUTPUT_ROOT = Path('/kaggle/working/attribute_test_runs' if EXECUTION_MODE == 'test_existing_last_blocks' else '/kaggle/working/attribute_module1_crop_runs')
PRECOMPUTE_RUN_ID = 'm1_inference_config_v1'
PRECOMPUTED_ROOT = Path('/kaggle/working/attribute_module1_precomputed') / PRECOMPUTE_RUN_ID
NOTEBOOK_REVISION = 'base_crop_reuse_preflight_v2'
# Luu cache dung resolution train de tranh day o dia Kaggle boi crop goc.
PRECOMPUTED_IMAGE_SIZE = 224
PREFLIGHT_SAMPLES_PER_TASK = 96
PREFLIGHT_SAFETY_FACTOR = 1.35
PREFLIGHT_RESERVED_SPACE_GIB = 4.0
ABORT_WHEN_FREE_SPACE_BELOW_GIB = 2.0
# 'current' giu RGB ROI production hien tai; 'masked' dung clean-mask crop nen xam candidate.
SHAPE_CONTRACT = 'masked'
PRECOMPUTED_SHAPE_DIR = PRECOMPUTED_ROOT / ('shape_masked' if SHAPE_CONTRACT == 'masked' else 'shape_current')
ENABLE_RANDOM_SHAPE_BACKGROUND = True
RANDOM_BACKGROUND_PROBABILITY = 0.50
REBUILD_PRECOMPUTED_DATASET = False
# Chi xoa cache precompute dang do neu khong co manifest hoan tat (vd. loi het disk).
CLEAR_INCOMPLETE_PRECOMPUTE = True
MODEL_ARTIFACT_DIR = find_model_artifact_dir() if EXECUTION_MODE == 'test_existing_last_blocks' else None

HEAD_RUN_ID = 'attr_head_v1'
LAST_RUN_ID = 'attr_last_blocks_v1'
RUN_HEAD_TRAIN = EXECUTION_MODE in {'train_head_and_last_blocks', 'train_head_only'}
RUN_HEAD_TEST = EXECUTION_MODE in {'train_head_and_last_blocks', 'train_head_only'}
RUN_LAST_BLOCKS_TRAIN = EXECUTION_MODE == 'train_head_and_last_blocks'
RUN_LAST_BLOCKS_TEST = EXECUTION_MODE in {'test_existing_last_blocks', 'train_head_and_last_blocks'}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if SHAPE_CONTRACT not in {'current', 'masked'}:
    raise ValueError('SHAPE_CONTRACT must be current or masked.')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('EXECUTION_MODE:', EXECUTION_MODE)
print('DATA_ROOT:', DATA_ROOT)
print('SEGMENTATION_WEIGHTS:', SEGMENTATION_WEIGHTS)
print('PRECOMPUTED_ROOT:', PRECOMPUTED_ROOT)
print('NOTEBOOK_REVISION:', NOTEBOOK_REVISION)
print('PRECOMPUTED_IMAGE_SIZE:', PRECOMPUTED_IMAGE_SIZE)
print('MODEL_ARTIFACT_DIR:', MODEL_ARTIFACT_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
# Preflight: kiem tra schema CSV, source group leakage va synthetic chi nam o train.
from pill_safety.cv.attribute.training.data_contract import validate_attribute_data

paths = {
    'shape_image_dir': DATA_ROOT / 'image_all/nih_attribute/shape',
    'color_image_dir': DATA_ROOT / 'image_all/nih_attribute/color',
    'label_mapping': DATA_ROOT / 'processed/nih_attribute/label_mapping.json',
    'shape_train_csv': DATA_ROOT / 'splits/nih_attribute/shape/train_combined_crop.csv',
    'shape_val_csv': DATA_ROOT / 'splits/nih_attribute/shape/val_combined_crop.csv',
    'shape_test_csv': DATA_ROOT / 'splits/nih_attribute/shape/test_combined_crop.csv',
    'color_train_csv': DATA_ROOT / 'splits/nih_attribute/color/train_multilabel.csv',
    'color_val_csv': DATA_ROOT / 'splits/nih_attribute/color/val_multilabel.csv',
    'color_test_csv': DATA_ROOT / 'splits/nih_attribute/color/test_multilabel.csv',
}
manifest = validate_attribute_data(paths, verify_images=True)
manifest

# Precompute Module 1 mot lan: DataLoader ve sau chi doc crop cache, khong chay YOLO moi batch.
# Dung nguyen configs/inference/segmentation.yaml; chi doi checkpoint va output artifact.
import hashlib
import re
import shutil
from dataclasses import replace

import pandas as pd
from tqdm.auto import tqdm

from pill_safety.cv.segmentation import SegmentationConfig, SegmentationPredictor
from pill_safety.schemas import SegmentationInferenceRequest

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def _save_training_crop(source_path: Path, destination_path: Path) -> None:
    # Transform goc cung resize ve 224. Cache JPEG q95/subsampling=0 cho ca shape va color
    # de giu full 57k samples trong quota Kaggle; khong luu crop goc hay artifact debug.
    with Image.open(source_path) as source:
        image = source.convert('RGB').resize(
            (PRECOMPUTED_IMAGE_SIZE, PRECOMPUTED_IMAGE_SIZE),
            Image.Resampling.BILINEAR,
        )
        image.save(destination_path, format='JPEG', quality=95, subsampling=0)

def _write_final_training_crop(instance, task: str, destination_path: Path) -> None:
    if task == 'shape':
        current_path = Path(instance.shape_crop_path)
        masked_path = current_path.with_name(
            current_path.name.replace('_shape_crop.png', '_shape_masked_control_crop.png')
        )
        if not masked_path.is_file():
            raise FileNotFoundError(f'Masked shape control crop missing: {masked_path}')
        source_path = current_path if SHAPE_CONTRACT == 'current' else masked_path
    else:
        source_path = Path(instance.color_crop_path)
    _save_training_crop(source_path, destination_path)

def _delete_intermediate_artifacts(instance) -> None:
    # Predictor ghi artifact theo image; sau khi da cache crop train thi xoa ngay.
    # Khong xoa PRECOMPUTED_ROOT hay crop cache cuoi cung.
    crop_directory = Path(instance.crop_path).parent
    request_directory, image_directory = crop_directory.parent.name, crop_directory.name
    for directory in (
        crop_directory,
        segmentation_config.output_dir / 'masks' / request_directory / image_directory,
        segmentation_config.output_dir / 'predictions' / 'segmentation' / request_directory / image_directory,
    ):
        shutil.rmtree(directory, ignore_errors=True)

precompute_manifest_path = PRECOMPUTED_ROOT / 'manifests' / 'precompute_manifest.json'
if (
    CLEAR_INCOMPLETE_PRECOMPUTE
    and PRECOMPUTED_ROOT.exists()
    and not precompute_manifest_path.is_file()
    and any(PRECOMPUTED_ROOT.iterdir())
):
    # Chi la output sinh boi notebook, khong phai DATA_ROOT attach tu Kaggle.
    print('Removing incomplete precompute cache:', PRECOMPUTED_ROOT)
    shutil.rmtree(PRECOMPUTED_ROOT)

if precompute_manifest_path.is_file() and not REBUILD_PRECOMPUTED_DATASET:
    precompute_manifest = json.loads(precompute_manifest_path.read_text(encoding='utf-8'))
    print('Reuse precomputed dataset:', PRECOMPUTED_ROOT)
elif PRECOMPUTED_ROOT.exists() and any(PRECOMPUTED_ROOT.iterdir()):
    raise FileExistsError(
        f'{PRECOMPUTED_ROOT} da ton tai. Doi PRECOMPUTE_RUN_ID thay vi ghi de cache cu.'
    )
else:
    for directory in (
        PRECOMPUTED_SHAPE_DIR,
        PRECOMPUTED_ROOT / 'color',
        PRECOMPUTED_ROOT / 'splits',
        PRECOMPUTED_ROOT / 'processed' / 'nih_attribute',
        PRECOMPUTED_ROOT / 'manifests',
    ):
        directory.mkdir(parents=True, exist_ok=True)

    segmentation_config = SegmentationConfig.from_yaml(
        REPO_DIR / 'configs' / 'inference' / 'segmentation.yaml'
    ).with_weights_path(SEGMENTATION_WEIGHTS).with_output_dir(
        PRECOMPUTED_ROOT / 'module1_artifacts'
    )
    # save_overlay chi la I/O artifact, khong thay doi mask hay crop inference.
    segmentation_config = replace(segmentation_config, save_overlay=False)
    segmentation_predictor = SegmentationPredictor(config=segmentation_config)
    rows, failures = [], []

    task_specs = {
        'shape': {
            'image_dir': paths['shape_image_dir'],
            'csvs': {split: paths[f'shape_{split}_csv'] for split in ('train', 'val', 'test')},
        },
        'color': {
            'image_dir': paths['color_image_dir'],
            'csvs': {split: paths[f'color_{split}_csv'] for split in ('train', 'val', 'test')},
        },
    }

    _OFFLINE_AUGMENT_SUFFIX = re.compile(r'_(?:AUG|ADV)_\d+(?=\.[^.]+$)', re.IGNORECASE)

    def _base_source_name(filename: str) -> str:
        # _AUG/_ADV la bien the offline cua anh goc. Module 1 chi segment anh goc;
        # Dataset van giu moi dong augment va tao bien the sau crop trong DataLoader.
        base_name = str(filename)
        while True:
            next_name = _OFFLINE_AUGMENT_SUFFIX.sub('', base_name)
            if next_name == base_name:
                return base_name
            base_name = next_name

    def _cache_filename(source_name: str) -> str:
        return f'{Path(_base_source_name(source_name)).stem}.jpg'

    precompute_plan = {}
    for task, spec in task_specs.items():
        trial_frames, base_sources = {}, []
        for split, csv_path in spec['csvs'].items():
            trial_frame = pd.read_csv(csv_path).copy()
            original_names = trial_frame['rximageFileName'].astype(str)
            base_names = original_names.map(_base_source_name)
            trial_frame['source_rximageFileName'] = original_names
            trial_frame['module1_source_rximageFileName'] = base_names
            trial_frame['rximageFileName'] = base_names.map(_cache_filename)
            trial_frames[split] = trial_frame
            base_sources.extend(base_names.tolist())
        source_names = sorted(set(base_sources))
        missing_sources = [name for name in source_names if not (spec['image_dir'] / name).is_file()]
        if missing_sources:
            raise FileNotFoundError(
                f'{task}: missing {len(missing_sources)} base images for offline variants, e.g. {missing_sources[:3]}'
            )
        precompute_plan[task] = {'trial_frames': trial_frames, 'source_names': source_names}
        print(
            f'{task}: keep {sum(len(frame) for frame in trial_frames.values())} training/eval rows; '
            f'run Module 1 on {len(source_names)} unique base images.'
        )

    def _free_space_gib() -> float:
        return shutil.disk_usage('/kaggle/working').free / (1024 ** 3)

    def _run_storage_preflight() -> dict:
        # Do dung luong crop sau dung codec/tranform that; neu khong du thi fail trong vai phut.
        probe_root = PRECOMPUTED_ROOT / '_storage_preflight'
        bytes_per_task, counts_per_task = {}, {}
        try:
            for task, spec in task_specs.items():
                source_names = precompute_plan[task]['source_names']
                counts_per_task[task] = len(source_names)
                probe = pd.Series(source_names).sample(
                    n=min(PREFLIGHT_SAMPLES_PER_TASK, len(source_names)), random_state=42
                ).reset_index(drop=True)
                sample_sizes = []
                for probe_index, probe_row in tqdm(
                    probe.items(), total=len(probe), desc=f'Storage preflight {task}'
                ):
                    source_name = str(probe_row)
                    request = SegmentationInferenceRequest(
                        request_id=f'attribute_storage_preflight_{task}',
                        session_id=PRECOMPUTE_RUN_ID,
                        image_id=f'{Path(source_name).stem}_{probe_index}',
                        image_path=str(spec['image_dir'] / source_name),
                    )
                    artifacts = segmentation_predictor.predict_with_artifacts(request)
                    if len(artifacts.output.instances) != 1:
                        raise RuntimeError(
                            f'Preflight {task}: expected exactly 1 pill for {source_name}, '
                            f'got {len(artifacts.output.instances)}'
                        )
                    instance = artifacts.output.instances[0]
                    destination = probe_root / task / f'{probe_index}.jpg'
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    try:
                        _write_final_training_crop(instance, task, destination)
                        sample_sizes.append(destination.stat().st_size)
                    finally:
                        _delete_intermediate_artifacts(instance)
                if not sample_sizes:
                    raise RuntimeError(f'Preflight {task} did not produce any cache sample.')
                bytes_per_task[task] = sum(sample_sizes) / len(sample_sizes)
            estimated_bytes = sum(bytes_per_task[task] * counts_per_task[task] for task in task_specs)
            estimate = {
                'samples_per_task': PREFLIGHT_SAMPLES_PER_TASK,
                'mean_bytes_per_task': {task: round(value) for task, value in bytes_per_task.items()},
                'estimated_cache_gib': round(estimated_bytes / (1024 ** 3), 3),
                'free_space_gib': round(_free_space_gib(), 3),
            }
            required_gib = estimate['estimated_cache_gib'] * PREFLIGHT_SAFETY_FACTOR + PREFLIGHT_RESERVED_SPACE_GIB
            estimate['required_free_space_gib'] = round(required_gib, 3)
            print('Storage preflight:', estimate)
            if estimate['free_space_gib'] < required_gib:
                raise RuntimeError(
                    'Khong du dung luong cho full-data cache. Da dung truoc precompute; '
                    f'can {required_gib:.2f} GiB, con {estimate["free_space_gib"]:.2f} GiB.'
                )
            return estimate
        finally:
            shutil.rmtree(probe_root, ignore_errors=True)

    storage_preflight = _run_storage_preflight()

    for task, spec in task_specs.items():
        source_names = precompute_plan[task]['source_names']
        iterator = tqdm(enumerate(source_names), total=len(source_names), desc=f'Precompute base {task}')
        for row_index, source_name in iterator:
            source_path = spec['image_dir'] / source_name
            try:
                request = SegmentationInferenceRequest(
                    request_id=f'attribute_precompute_base_{task}',
                    session_id=PRECOMPUTE_RUN_ID,
                    image_id=f'{Path(source_name).stem}_{row_index}',
                    image_path=str(source_path),
                )
                segmentation_artifacts = segmentation_predictor.predict_with_artifacts(request)
                instances = segmentation_artifacts.output.instances
                if len(instances) != 1:
                    raise RuntimeError(f'expected exactly 1 pill, got {len(instances)}')
                instance = instances[0]
                output_name = _cache_filename(source_name)
                destination = PRECOMPUTED_SHAPE_DIR / output_name if task == 'shape' else PRECOMPUTED_ROOT / 'color' / output_name
                _write_final_training_crop(instance, task, destination)
                _delete_intermediate_artifacts(instance)
                rows.append({
                    'task': task, 'source': source_name, 'output': output_name,
                    'segmentation_confidence': float(instance.segmentation.confidence),
                    'quality_flags': list(instance.quality_flags),
                })
                if len(rows) % 250 == 0:
                    free_space_gib = _free_space_gib()
                    print(f'Cached {len(rows)} base crops | free disk: {free_space_gib:.2f} GiB')
                    if free_space_gib < ABORT_WHEN_FREE_SPACE_BELOW_GIB:
                        raise RuntimeError(
                            f'Dung luong con {free_space_gib:.2f} GiB, duoi nguong an toan '
                            f'{ABORT_WHEN_FREE_SPACE_BELOW_GIB:.2f} GiB. Da dung truoc khi disk day.'
                        )
            except Exception as error:
                failures.append({
                    'task': task, 'source': source_name,
                    'error': f'{type(error).__name__}: {error}',
                })

        for split, trial_frame in precompute_plan[task]['trial_frames'].items():
            trial_frame.to_csv(PRECOMPUTED_ROOT / 'splits' / f'{task}_{split}.csv', index=False)

    if failures:
        failure_path = PRECOMPUTED_ROOT / 'manifests' / 'precompute_failures.json'
        failure_path.write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding='utf-8')
        raise RuntimeError(
            f'Module 1 failed on {len(failures)} samples; cache khong day du nen khong train. '
            f'Xem {failure_path} de sua data/segmentation truoc.'
        )

    shutil.copy2(
        paths['label_mapping'], PRECOMPUTED_ROOT / 'processed' / 'nih_attribute' / 'label_mapping.json'
    )
    precompute_manifest = {
        'run_id': PRECOMPUTE_RUN_ID,
        'source_data_root': str(DATA_ROOT),
        'segmentation_checkpoint': str(SEGMENTATION_WEIGHTS),
        'segmentation_checkpoint_sha256': _sha256(SEGMENTATION_WEIGHTS),
        'segmentation_config_source': str(REPO_DIR / 'configs' / 'inference' / 'segmentation.yaml'),
        'segmentation_config_sha256': _sha256(REPO_DIR / 'configs' / 'inference' / 'segmentation.yaml'),
        'runtime_overrides': {'save_overlay': False},
        'cache_image_size': PRECOMPUTED_IMAGE_SIZE,
        'shape_contract': SHAPE_CONTRACT,
        'cache_format': 'base_image_jpeg_quality_95_subsampling_0',
        'augmentation_contract': 'all original CSV rows retained; offline AUG/ADV rows reuse base Module 1 crop and receive online post-crop augmentation',
        'rows_retained_per_task': {task: sum(len(frame) for frame in plan['trial_frames'].values()) for task, plan in precompute_plan.items()},
        'unique_module1_inputs_per_task': {task: len(plan['source_names']) for task, plan in precompute_plan.items()},
        'storage_preflight': storage_preflight,
        'intermediate_artifacts': 'deleted after each sample',
        'successful_samples': rows,
        'failures': failures,
    }
    precompute_manifest_path.write_text(
        json.dumps(precompute_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print('Precompute complete:', PRECOMPUTED_ROOT)

print('Precompute summary:', {
    'successful': len(precompute_manifest['successful_samples']),
    'failures': len(precompute_manifest['failures']),
    'manifest': str(precompute_manifest_path),
})

In [ ]:
import json

from pill_safety.cv.attribute.training.workflow import (
    calibrate_color_thresholds,
    compare_validation_runs,
    evaluate_test,
    load_config,
    train,
)

HEAD_CONFIG = load_config(REPO_DIR / 'configs/training/attribute_resnet18_head_tune/config.yaml')
# Test-only dung dung model_config di kem checkpoint; train thi dung config train cua repo.
LAST_CONFIG = load_config(
    MODEL_ARTIFACT_DIR / 'model_config.yaml'
    if EXECUTION_MODE == 'test_existing_last_blocks'
    else REPO_DIR / 'configs/training/attribute_resnet18_last_blocks_finetune/config.yaml'
)

# Trial config tro vao cache Module 1; source workflow, labels va losses giu nguyen.
from copy import deepcopy
from torch.utils.data import DataLoader
from torchvision import transforms
from pill_safety.cv.attribute.datasets.color_dataset import ColorDataset
from pill_safety.cv.attribute.datasets.shape_dataset import ShapeDataset
from pill_safety.cv.attribute.utils.transforms import get_color_transforms, get_shape_transforms
import pill_safety.cv.attribute.training.workflow as training_workflow

TRIAL_SHAPE_DIR = PRECOMPUTED_ROOT / ('shape_masked' if SHAPE_CONTRACT == 'masked' else 'shape_current')
TRIAL_COLOR_DIR = PRECOMPUTED_ROOT / 'color'

def build_trial_config(base_config):
    config = deepcopy(base_config)
    config['data']['root'] = str(PRECOMPUTED_ROOT)
    config['data']['label_mapping'] = 'processed/nih_attribute/label_mapping.json'
    config['data']['shape']['image_dir'] = TRIAL_SHAPE_DIR.relative_to(PRECOMPUTED_ROOT).as_posix()
    config['data']['color']['image_dir'] = TRIAL_COLOR_DIR.relative_to(PRECOMPUTED_ROOT).as_posix()
    for split in ('train', 'val', 'test'):
        config['data']['shape'][f'{split}_csv'] = f'splits/shape_{split}.csv'
        config['data']['color'][f'{split}_csv'] = f'splits/color_{split}.csv'
    return config

TRIAL_HEAD_CONFIG = build_trial_config(HEAD_CONFIG)
TRIAL_LAST_CONFIG = build_trial_config(LAST_CONFIG)
trial_paths = training_workflow.resolve_paths(TRIAL_LAST_CONFIG, str(PRECOMPUTED_ROOT), str(OUTPUT_ROOT))[0]
trial_manifest = validate_attribute_data(trial_paths, verify_images=True)
display({'shape_contract': SHAPE_CONTRACT, 'shape_dir': str(TRIAL_SHAPE_DIR), 'color_dir': str(TRIAL_COLOR_DIR), 'trial_manifest': trial_manifest})

def _trial_shape_transform(image_size, synthetic: bool):
    if not synthetic:
        return get_shape_transforms(image_size)
    return transforms.Compose([
        transforms.RandomAffine(degrees=15, translate=(0.06, 0.06), scale=(0.92, 1.08), shear=5, fill=(127, 127, 127)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def _trial_color_transform(image_size, synthetic: bool):
    if not synthetic:
        return get_color_transforms(image_size)
    return transforms.Compose([
        transforms.RandomAffine(degrees=12, translate=(0.04, 0.04), scale=(0.94, 1.06), fill=(127, 127, 127)),
        transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

class TrialShapeDataset(ShapeDataset):
    def __init__(self, csv_file, img_dir, image_size, random_background):
        super().__init__(csv_file, img_dir, transform=None)
        self.base_transform = _trial_shape_transform(image_size, synthetic=False)
        self.synthetic_transform = _trial_shape_transform(image_size, synthetic=True)
        self.random_background = random_background

    def __getitem__(self, idx):
        row = self.data_frame.iloc[int(idx)]
        path = Path(self.img_dir) / Path(str(row[self.image_col])).name
        with Image.open(path) as source:
            image = source.convert('RGB')
        if self.random_background and np.random.random() < RANDOM_BACKGROUND_PROBABILITY:
            array = np.asarray(image, dtype=np.uint8).copy()
            # JPEG co the lam canvas 127 lech vai don vi gan bien; dung tolerance nho.
            canvas = np.max(np.abs(array.astype(np.int16) - 127), axis=2) <= 3
            height, width = canvas.shape
            base = np.random.randint(35, 220, size=(1, 1, 3)).astype(np.float32)
            gradient = np.linspace(-35, 35, width, dtype=np.float32)[None, :, None]
            noise = np.random.normal(0, 9, size=(height, width, 1))
            background = np.clip(base + gradient + noise, 0, 255).astype(np.uint8)
            array[canvas] = background[canvas]
            image = Image.fromarray(array, mode='RGB')
        is_synthetic = str(row.get('source_rximageFileName', '')) != str(row.get('module1_source_rximageFileName', ''))
        transform = self.synthetic_transform if is_synthetic else self.base_transform
        return transform(image), int(row[self.label_col])

class TrialColorDataset(ColorDataset):
    def __init__(self, csv_file, img_dir, image_size):
        super().__init__(csv_file, img_dir, transform=None)
        self.base_transform = _trial_color_transform(image_size, synthetic=False)
        self.synthetic_transform = _trial_color_transform(image_size, synthetic=True)

    def __getitem__(self, idx):
        row = self.df.iloc[int(idx)]
        path = Path(self.img_dir) / Path(str(row['rximageFileName'])).name
        with Image.open(path) as source:
            image = source.convert('RGB')
        transform = self.synthetic_transform if int(row.get('is_synthetic', 0)) == 1 else self.base_transform
        labels = torch.tensor([float(row[column]) for column in self.color_cols], dtype=torch.float32)
        return transform(image), labels

def build_module1_trial_loaders(paths, config, split, shuffle):
    image_size = config['training']['image_size']
    workers = config['training'].get('num_workers', 2)
    pin_memory = torch.cuda.is_available()
    shape_set = TrialShapeDataset(paths[f'shape_{split}_csv'], paths['shape_image_dir'], image_size, split == 'train' and ENABLE_RANDOM_SHAPE_BACKGROUND)
    color_set = TrialColorDataset(paths[f'color_{split}_csv'], paths['color_image_dir'], image_size)
    return (
        DataLoader(shape_set, batch_size=config['sampling']['shape_batch_size'], shuffle=shuffle, num_workers=workers, pin_memory=pin_memory),
        DataLoader(color_set, batch_size=config['sampling']['color_batch_size'], shuffle=shuffle, num_workers=workers, pin_memory=pin_memory),
    )

# Override runtime trong notebook: train/evaluate/calibrate deu dung trial loaders, source khong bi sua.
training_workflow._build_loaders = build_module1_trial_loaders
print('Module 1 trial loader override installed.')

In [ ]:
# Phase 1 chi can khi muon train/test head. Test-only last-block bo qua toan bo pha nay.
head_checkpoint = OUTPUT_ROOT / 'attribute_resnet18_head_tune/checkpoints' / f'{HEAD_RUN_ID}_best.pt'
head_result = None
if RUN_HEAD_TRAIN:
    head_result = train(TRIAL_HEAD_CONFIG, str(PRECOMPUTED_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    head_checkpoint = Path(head_result['checkpoint'])
if RUN_HEAD_TRAIN or RUN_HEAD_TEST:
    if not head_checkpoint.is_file():
        raise FileNotFoundError(f'Head checkpoint not found: {head_checkpoint}')
    head_result = {'checkpoint': str(head_checkpoint)}
    head_result
else:
    print('Head phase skipped: test-only last-block mode.')

In [ ]:
# Chi calibrate head khi head duoc train/test. Test-only last-block khong dong vao validation cua head.
head_threshold_result = None
head_thresholds = None
if RUN_HEAD_TRAIN or RUN_HEAD_TEST:
    head_threshold_result = calibrate_color_thresholds(TRIAL_HEAD_CONFIG, head_checkpoint, str(PRECOMPUTED_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    head_thresholds = Path(head_threshold_result['path'])
    head_threshold_result
else:
    print('Head calibration skipped: test-only last-block mode.')

In [ ]:
# Test chi la reporting sau khi checkpoint va threshold da duoc chon bang validation.
# Test khong duoc dung de tune threshold, chon epoch hay chon chien luoc fine-tune.
if RUN_HEAD_TEST:
    assert head_threshold_result['split'] == 'validation'
    head_test = evaluate_test(TRIAL_HEAD_CONFIG, head_checkpoint, head_thresholds, str(PRECOMPUTED_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    assert head_test['split'] == 'test'
    print('=== HEAD-TUNE: TEST METRICS ===')
    print(json.dumps(head_test['metrics'], indent=2))
    print('Per-class shape F1:', json.dumps(head_test['per_class_metrics']['shape'], indent=2))
    print('Per-color F1:', json.dumps(head_test['per_class_metrics']['color'], indent=2))
    print('Test metric file:', head_test['path'])
    print('Test plots:', Path(head_test['path']).parent.parent / 'plots')
    print('Prediction examples:', Path(head_test['path']).parent.parent / 'predictions' / HEAD_RUN_ID)
else:
    print('Head test skipped: RUN_HEAD_TEST=False')

In [ ]:
# Phase 2: test-only nap thang best.pt da dong goi; neu train thi nap head best va fine-tune layer3/layer4.
last_checkpoint = None
last_result = None
if RUN_LAST_BLOCKS_TRAIN or RUN_LAST_BLOCKS_TEST:
    last_checkpoint = (
        MODEL_ARTIFACT_DIR / 'best.pt'
        if EXECUTION_MODE == 'test_existing_last_blocks'
        else OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/checkpoints' / f'{LAST_RUN_ID}_best.pt'
    )
    if RUN_LAST_BLOCKS_TRAIN:
        if not head_checkpoint.is_file():
            raise FileNotFoundError(f'Head checkpoint required for last-block training: {head_checkpoint}')
        last_result = train(TRIAL_LAST_CONFIG, str(PRECOMPUTED_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID, pretrained_override=str(head_checkpoint))
        last_checkpoint = Path(last_result['checkpoint'])
    if not last_checkpoint.is_file():
        raise FileNotFoundError(f'Last-block checkpoint not found: {last_checkpoint}')

    # Test-only phai kiem tra model va test data dung cung label order.
    if EXECUTION_MODE == 'test_existing_last_blocks':
        artifact_mapping = json.loads((MODEL_ARTIFACT_DIR / 'label_mapping.json').read_text(encoding='utf-8'))
        data_mapping = json.loads((DATA_ROOT / 'processed/nih_attribute/label_mapping.json').read_text(encoding='utf-8'))
        if artifact_mapping != data_mapping:
            raise ValueError('label_mapping cua model artifact khong khop label_mapping cua test dataset.')
    last_result = {'checkpoint': str(last_checkpoint), 'mode': 'train' if RUN_LAST_BLOCKS_TRAIN else 'test_only'}
    last_result
else:
    print('Last-block phase skipped: head-only training mode.')

In [ ]:
# Test-only dung nguyen threshold da calibrate tren validation va dong goi cung model.
# Khong chay lai calibration tren validation de giu nguyen model artifact duoc danh gia.
if RUN_LAST_BLOCKS_TRAIN:
    last_threshold_result = calibrate_color_thresholds(TRIAL_LAST_CONFIG, last_checkpoint, str(PRECOMPUTED_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
    last_thresholds = Path(last_threshold_result['path'])
elif RUN_LAST_BLOCKS_TEST:
    last_thresholds = MODEL_ARTIFACT_DIR / 'optimal_thresholds.json'
    if not last_thresholds.is_file():
        raise FileNotFoundError(f'Optimal thresholds not found: {last_thresholds}')
    last_threshold_result = json.loads(last_thresholds.read_text(encoding='utf-8'))
else:
    last_thresholds = None
    last_threshold_result = None
    print('Last-block calibration/test skipped: head-only training mode.')

if RUN_LAST_BLOCKS_TEST:
    assert last_threshold_result['split'] == 'validation'
    last_test = evaluate_test(TRIAL_LAST_CONFIG, last_checkpoint, last_thresholds, str(PRECOMPUTED_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
    assert last_test['split'] == 'test'
    print('=== LAST-BLOCKS: TEST METRICS ===')
    print(json.dumps(last_test['metrics'], indent=2))
    print('Per-class shape F1:', json.dumps(last_test['per_class_metrics']['shape'], indent=2))
    print('Per-color F1:', json.dumps(last_test['per_class_metrics']['color'], indent=2))
    print('Test metric file:', last_test['path'])
    print('Test plots:', Path(last_test['path']).parent.parent / 'plots')
    print('Prediction examples:', Path(last_test['path']).parent.parent / 'predictions' / LAST_RUN_ID)
else:
    print('Last-block test skipped: RUN_LAST_BLOCKS_TEST=False')

In [ ]:
# Chi so sanh/selection khi notebook vua train ca head va last-block. Test-only khong phat sinh selection moi.
if RUN_HEAD_TRAIN and RUN_LAST_BLOCKS_TRAIN:
    comparison = compare_validation_runs(
        OUTPUT_ROOT / 'attribute_resnet18_head_tune/metrics' / f'{HEAD_RUN_ID}_val_metrics.json',
        OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/metrics' / f'{LAST_RUN_ID}_val_metrics.json',
        OUTPUT_ROOT / 'attribute_model_selection.json',
    )
    comparison
else:
    print('Validation model selection skipped: test-only uses the supplied last-block checkpoint.')